In [ ]:
!pip install openai

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!python --version


Python 3.10.12


In [ ]:
pip install openai==0.28

In [ ]:
!pip install tqdm

In [ ]:
import json
# Load existing dataset
with open('/content/drive/MyDrive/remain/tasks_without_questions.json', 'r') as file:
    existing_data = json.load(file)

In [ ]:
all_video_ids = []
for video_id, video_info in existing_data['database'].items():
  all_video_ids.append(video_id)

In [ ]:
from tqdm import tqdm
for id in tqdm(all_video_ids, desc="Generating New Data for MTP", total=len(all_video_ids)):
  print()
  print(id)
  video_info = existing_data['database'][id]
  print(video_info)
  task_name = video_info['class']  # Using 'class' as task name
  steps = [annotation['label'] for annotation in video_info['annotation']]
  print(task_name)
  print()
  print(steps)
  break

Generating New Data for MTP:   0%|          | 0/2 [00:00<?, ?it/s]


XRW6xIlarME
{'recipe_type': 100, 'annotation': [{'id': '308', 'segment': [23.0, 27.0], 'label': 'prepare and boil water'}, {'id': '310', 'segment': [46.0, 51.0], 'label': 'add some water to the tea'}, {'id': '312', 'segment': [55.0, 64.0], 'label': 'pour the tea into the vessel'}, {'id': '310', 'segment': [74.0, 76.0], 'label': 'add some water to the tea'}, {'id': '312', 'segment': [81.0, 85.0], 'label': 'pour the tea into the vessel'}], 'video_url': 'https://www.youtube.com/embed/XRW6xIlarME', 'start': 16.798959768048086, 'end': 91.6306577000547, 'duration': 94.273011, 'class': 'MakeTea', 'subset': 'training', 'verification_data': []}
MakeTea

['prepare and boil water', 'add some water to the tea', 'pour the tea into the vessel', 'add some water to the tea', 'pour the tea into the vessel']


**trying the code in perplexity**

In [ ]:
import json
import openai
from tqdm import tqdm

# Your OpenAI API key
openai.api_key = 'api key'

def generate_verification_questions(task_name, steps):
    steps_formatted = "\n".join([f'"{step}"' for step in steps])  # Format the steps
    prompt = f"""
    Here are the steps in chronological order for the task of {task_name}:
    {steps_formatted}

    Please generate one concise verification question and its corresponding answer for each step. Ensure that you provide a question and answer for every step listed above. Each question should assess the correctness of its corresponding step. Use 'What', 'How', 'Did', or 'When' to start the questions. Format the output as:
    Question: "<question>"
    Answer: "<answer>"
    """

    # Call the OpenAI API to get the questions and answers
    response = openai.ChatCompletion.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=16000  # Increased token limit for more output
    )

    # Debugging output
    # print("API Response:", response['choices'][0]['message']['content'])

    return response['choices'][0]['message']['content']

# Load existing dataset
with open('/content/drive/MyDrive/remain/tasks_without_questions.json', 'r') as file:
    existing_data = json.load(file)

new_data = {
    "database": {}
}

for video_id in tqdm(all_video_ids, desc="Generating New Data for MTP", total=len(all_video_ids)):
    video_info = existing_data['database'][video_id]
    task_name = video_info['class']  # Using 'class' as task name
    steps = [annotation['label'] for annotation in video_info['annotation']]  # Extracting labels

    # Generate verification questions and answers
    verification_data = generate_verification_questions(task_name, steps)

    # Split the verification_data into structured format
    verification_entries = []
    lines = verification_data.strip().split('\n')

    # Initialize variables to capture questions and answers
    current_question = None

    # Loop through each line and capture questions and answers
    for line in lines:
        line = line.strip()
        if line.startswith("Question:"):
            if current_question is not None:  # Save previous question-answer pair if exists
                verification_entries.append(current_question)
            current_question = {"question": line.replace('Question: ', '').strip('" ')}
        elif line.startswith("Answer:") and current_question is not None:
            current_question["answer"] = line.replace('Answer: ', '').strip('" ')
            verification_entries.append(current_question)
            current_question = None  # Reset for next pair

    # Add verification entries to the video info
    video_info['verification_data'] = verification_entries

    # Append the updated video info to the new dataset
    new_data['database'][video_id] = video_info

# Save the new dataset
with open('/content/drive/MyDrive/remain/tasks_without_questions.json', 'w') as file:
    json.dump(new_data, file, indent=4)

print("New dataset created successfully!")

Generating New Data for MTP: 100%|██████████| 2/2 [00:08<00:00,  4.05s/it]

New dataset created successfully!


In [ ]:
from graphviz import Digraph

def create_pipeline_visualization():
    # Create a new directed graph
    dot = Digraph(comment='Video QA Pipeline Architecture')

    # Graph settings
    dot.attr(rankdir='LR')  # Left to right layout
    dot.attr('node', shape='box', style='rounded,filled', fontname='Arial')

    # Color scheme
    colors = {
        'input': '#E6F3FF',       # Light blue
        'feature': '#FFE6E6',     # Light red
        'temporal': '#E6FFE6',    # Light green
        'fusion': '#FFE6FF',      # Light purple
        'output': '#FFFFF0'       # Light yellow
    }

    # Input nodes
    with dot.subgraph(name='cluster_0') as c:
        c.attr(label='Input Processing', style='rounded', bgcolor=colors['input'])
        c.node('video', 'Video Input', fillcolor='#ADD8E6')
        c.node('frames', 'Frame Extraction', fillcolor='#ADD8E6')
        c.node('question', 'Step Annotations & Question Input', fillcolor='#ADD8E6')
        # c.node('annot', 'Step Annotations', fillcolor='#ADD8E6')

    # Feature extraction
    with dot.subgraph(name='cluster_1') as c:
        c.attr(label='Feature Extraction', style='rounded', bgcolor=colors['feature'])
        c.node('clip', 'CLIP\nVisual Features', fillcolor='#FFB6C1')
        c.node('text_embed', 'Text Embedding', fillcolor='#FFB6C1')

    # Temporal processing
    with dot.subgraph(name='cluster_2') as c:
        c.attr(label='Temporal Processing', style='rounded', bgcolor=colors['temporal'])
        c.node('videobert', 'VideoBERT\nTemporal Modeling', fillcolor='#98FB98')

    # Feature fusion
    with dot.subgraph(name='cluster_3') as c:
        c.attr(label='Multimodal Fusion', style='rounded', bgcolor=colors['fusion'])
        c.node('fusion', 'Cross-Modal\nEncoder', fillcolor='#DDA0DD')
        c.node('projection', 'Projection Layer', fillcolor='#DDA0DD')

    # Answer generation
    with dot.subgraph(name='cluster_4') as c:
        c.attr(label='Answer Generation', style='rounded', bgcolor=colors['output'])
        c.node('t5', 'T5 Model', fillcolor='#F0E68C')
        c.node('answer', 'Generated Answer', fillcolor='#F0E68C')

    # Add edges
    dot.edge('video', 'frames')
    dot.edge('frames', 'clip')
    dot.edge('clip', 'videobert')
    dot.edge('question', 'text_embed')
    dot.edge('annot', 'text_embed')
    dot.edge('text_embed', 'fusion')
    dot.edge('videobert', 'fusion')
    dot.edge('fusion', 'projection')
    dot.edge('projection', 't5')
    dot.edge('t5', 'answer')

    # Save the visualization
    dot.render('vqa_pipeline', format='png', cleanup=True)

    return dot

# Create the visualization
pipeline = create_pipeline_visualization()

In [ ]:
from graphviz import Digraph

dot = Digraph(comment="Video QA Pipeline Summary", format="png")

# Step 1: Feature Extraction & Fusion
dot.node("A", "Raw Video Frames")
dot.node("B", "CLIP: Frame Features")
dot.node("C", "VideoBERT: Temporal Modeling")
dot.node("D", "Cross-Modal Fusion\n(Video + Task Steps)")
dot.edge("A", "B", label="Extract Frames")
dot.edge("B", "C", label="Temporal Features")
dot.edge("C", "D", label="Fuse with Text")

# Step 2: Convert Fused Features into Prompt Tokens
dot.node("E", "Projection Layer\n(Fused → Prompt Tokens)")
dot.edge("D", "E")

# Step 3: Answer Generation
dot.node("F", "Concatenate Prompt Tokens\n+ Question Tokens")
dot.node("G", "T5: Answer Generation")
dot.node("H", "Final Answer")
dot.edge("E", "F")
dot.edge("F", "G")
dot.edge("G", "H")

# Render the simplified summary diagram
dot.render("summary_pipeline.gv", view=True)


'summary_pipeline.gv.png'

In [ ]:
from graphviz import Digraph

dot = Digraph(comment="Alternate Visual Summary of Video QA Pipeline", format="png")
dot.attr(rankdir="LR", splines="ortho")  # Left-to-right layout with orthogonal edges

# Cluster 1: Feature Extraction & Fusion
with dot.subgraph(name="cluster_0") as c:
    c.attr(style="filled", color="lightgrey", label="Feature Extraction & Fusion")
    c.node("A", "Raw Video Frames")
    c.node("B", "CLIP\n(Frame Features)")
    c.node("C", "VideoBERT\n(Temporal Modeling)")
    c.node("D", "Cross-Modal Fusion\n(Video + Text Steps)")
    c.edge("A", "B", label="Extract Frames\n(1-2 fps)")
    c.edge("B", "C", label="Frame Embeddings")
    c.edge("C", "D", label="Temporal Features")

# Cluster 2: Prompt Token Generation
with dot.subgraph(name="cluster_1") as c:
    c.attr(style="filled", color="lightblue", label="Prompt Token Generation")
    c.node("E", "Projection Layer\n(Fused → Prompt Tokens)")

# Cluster 3: Answer Generation
with dot.subgraph(name="cluster_2") as c:
    c.attr(style="filled", color="lightyellow", label="Answer Generation")
    c.node("F", "Combine Prompt\n+ Question Tokens")
    c.node("G", "T5\n(Answer Generation)")
    c.node("H", "Post-Processing\n(Decode to Text)")
    c.edge("F", "G")
    c.edge("G", "H")

# Inter-cluster connections
dot.edge("D", "E", label="Fused Features")
dot.edge("E", "F", label="Visual Prompt Tokens")

# Render and view the graph (output file: 'alternate_pipeline.gv.png')
dot.render("alternate_pipeline.gv", view=True)


'alternate_pipeline.gv.png'

In [ ]:
from graphviz import Digraph

dot = Digraph(comment="Rectangular Layout for Video QA Pipeline", format="png")
# Set top-to-bottom layout and force a square canvas
dot.attr(rankdir="TB", size="8,8", ratio="fill", splines="ortho")

# Cluster 1: Feature Extraction & Fusion
with dot.subgraph(name="cluster_feature") as cf:
    cf.attr(label="Feature Extraction & Fusion", style="filled", color="lightgrey")
    cf.node("A", "Raw Video Frames")
    cf.node("B", "CLIP: Frame Features)")
    cf.node("C", "VideoBERT: Temporal Modeling")
    cf.node("D", "Cross-Modal Fusion\n(Video + Text Steps)")
    # Force nodes in this cluster to be on the same horizontal row
    cf.attr(rank="same")
    cf.edge("A", "B", arrowhead="vee")
    cf.edge("B", "C", arrowhead="vee")
    cf.edge("C", "D", arrowhead="vee")

# Cluster 2: Prompt Token Generation
with dot.subgraph(name="cluster_prompt") as cp:
    cp.attr(label="Prompt Token Generation", style="filled", color="lightblue")
    cp.node("E", "Projection Layer\n(Fused → Prompt Tokens)")
    # No rank constraint here since it's a single node

# Cluster 3: Answer Generation
with dot.subgraph(name="cluster_answer") as ca:
    ca.attr(label="Answer Generation", style="filled", color="lightyellow")
    ca.node("F", "Combine Prompt\n+ Question Tokens")
    ca.node("G", "T5: Answer Generation")
    ca.node("H", "Post-Processing\n(Decode to Text)")
    # Force answer nodes to appear in one row
    ca.attr(rank="same")
    ca.edge("F", "G", arrowhead="vee")
    ca.edge("G", "H", arrowhead="vee")

# Connect clusters vertically
dot.edge("D", "E", label="Fused Features", constraint="true")
dot.edge("E", "F", label="Visual Prompt Tokens", constraint="true")

# Render and display the diagram (output file: rectangular_pipeline.gv.png)
dot.render("rectangular_pipeline.gv", view=True)


'rectangular_pipeline.gv.png'

In [ ]:
from graphviz import Digraph

dot = Digraph(comment="Video QA Pipeline", format="png")
dot.attr(rankdir="TB")  # top-to-bottom flow

# ------------------
# Subgraph: Input
# ------------------
with dot.subgraph(name="cluster_input") as inp:
    inp.attr(label="Input", style="filled", color="lightgrey")
    inp.node("V", "Video")
    inp.node("FE", "Frame Extraction")
    inp.node("CLIP", "CLIP")
    inp.node("Q", "Question")
    inp.node("TK", "Tokenizer")
    inp.node("A", "Annotations")
    inp.node("AT", "Annotation Tokenizer")

    # Edges for video processing
    inp.edge("V", "FE")
    inp.edge("FE", "CLIP")
    # Edges for text inputs
    inp.edge("Q", "TK")
    inp.edge("A", "AT")

# ---------------------------
# Subgraph: Feature Processing
# ---------------------------
with dot.subgraph(name="cluster_feature_processing") as fp:
    fp.attr(label="Feature Processing", style="filled", color="lightblue")
    fp.node("CF", "CLIP Features")
    fp.node("VB", "VideoBERT")
    fp.node("TE", "Temporal Embeddings")

    fp.edge("CLIP", "CF")
    fp.edge("CF", "VB")
    fp.edge("VB", "TE")

# --------------------------
# Subgraph: Feature Fusion
# --------------------------
with dot.subgraph(name="cluster_feature_fusion") as ff:
    ff.attr(label="Feature Fusion", style="filled", color="lightyellow")
    ff.node("CM", "Cross-Modal Encoder")
    ff.node("PL", "Projection Layer")

    ff.edge("TE", "CM")
    ff.edge("TK", "CM")
    ff.edge("AT", "CM")
    ff.edge("CM", "PL")

# ---------------------------
# Subgraph: Answer Generation
# ---------------------------
with dot.subgraph(name="cluster_answer_generation") as ag:
    ag.attr(label="Answer Generation", style="filled", color="lightgreen")
    ag.node("PT", "Prompt Tokens")
    ag.node("CON", "Concatenation")
    ag.node("T5", "T5 Model")
    ag.node("PP", "Post-Processing")
    ag.node("ANS", "Final Answer")

    ag.edge("PL", "PT")
    ag.edge("PT", "CON")
    # Reusing the same question tokenizer output for concatenation
    ag.edge("TK", "CON")
    ag.edge("CON", "T5")
    ag.edge("T5", "PP")
    ag.edge("PP", "ANS")

# Render and display the diagram (output file: 'video_qa_pipeline.gv.png')
dot.render("video_qa_pipeline.gv", view=True)


'video_qa_pipeline.gv.png'

In [ ]:
from graphviz import Digraph

dot = Digraph(comment="Concise Video QA Pipeline", format="png")
dot.attr(rankdir="TB", splines="ortho")  # Top-to-bottom layout

# --- Input Subgraph ---
with dot.subgraph(name="cluster_input") as inp:
    inp.attr(label="Input", style="filled", color="lightgrey")
    inp.node("V", "Raw Video")
    inp.node("T", "Tokenized Q & Annotations")
    # Invisible edge to keep nodes aligned
    inp.edge("V", "T", style="invis")

# --- Feature Processing Subgraph ---
with dot.subgraph(name="cluster_processing") as proc:
    proc.attr(label="Feature Processing", style="filled", color="lightblue")
    proc.node("CL", "CLIP\n(Frame Features)")
    proc.node("VB", "VideoBERT\n(Temporal Modeling)")
    proc.edge("CL", "VB", arrowhead="vee")

# --- Feature Fusion Subgraph ---
with dot.subgraph(name="cluster_fusion") as fus:
    fus.attr(label="Feature Fusion", style="filled", color="lightyellow")
    fus.node("CM", "Cross-Modal Encoder\n(Fuse Video & Text)")
    fus.node("PL", "Projection Layer\n(Prompt Tokens)")
    fus.edge("CM", "PL", arrowhead="vee")

# --- Answer Generation Subgraph ---
with dot.subgraph(name="cluster_generation") as gen:
    gen.attr(label="Answer Generation", style="filled", color="lightgreen")
    gen.node("CON", "Concat Prompt & Q Tokens")
    gen.node("T5", "T5 Model\n(Answer Generation)")
    gen.node("ANS", "Final Answer")
    gen.edge("CON", "T5", arrowhead="vee")
    gen.edge("T5", "ANS", arrowhead="vee")

# --- Inter-Cluster Connections ---
dot.edge("V", "CL", label="Extract Features")
dot.edge("VB", "CM", label="Temporal Embeddings")
dot.edge("T", "CM", label="Textual Info", style="dashed")
dot.edge("PL", "CON", label="Visual Prompt Tokens")
dot.edge("T", "CON", label="Q Tokens", style="dashed")

# Render and view the diagram (output file: 'concise_video_qa_pipeline.gv.png')
dot.render("concise_video_qa_pipeline.gv", view=True)


'concise_video_qa_pipeline.gv.png'

In [1]:
from graphviz import Digraph

dot = Digraph(comment="Attractive Concise Video QA Pipeline", format="png")
dot.attr(rankdir="TB", splines="polyline", fontsize="12", fontname="Helvetica")

# Global node style
dot.attr('node', shape='box', style='filled', fontname="Helvetica", fontsize="10")

# --- Input Subgraph ---
with dot.subgraph(name="cluster_input") as inp:
    inp.attr(label="Input", style="rounded,filled", color="deepskyblue", fillcolor="lightcyan")
    inp.node("V", "Raw Video", fillcolor="white")
    inp.node("T", "Tokenized Q & Annotations", fillcolor="white")
    # Invisible edge to keep nodes aligned
    inp.edge("V", "T", style="invis")

# --- Feature Processing Subgraph ---
with dot.subgraph(name="cluster_processing") as proc:
    proc.attr(label="Feature Processing", style="rounded,filled", color="goldenrod", fillcolor="lightyellow")
    proc.node("CL", "CLIP\n(Frame Features)", fillcolor="white")
    proc.node("VB", "VideoBERT\n(Temporal Modeling)", fillcolor="white")
    proc.edge("CL", "VB", arrowhead="vee", color="goldenrod")

# --- Feature Fusion Subgraph ---
with dot.subgraph(name="cluster_fusion") as fus:
    fus.attr(label="Feature Fusion", style="rounded,filled", color="darkorchid", fillcolor="lavender")
    fus.node("CM", "Cross-Modal Encoder\n(Fuse Video & Text)", fillcolor="white")
    fus.node("PL", "Projection Layer\n(Prompt Tokens)", fillcolor="white")
    fus.edge("CM", "PL", arrowhead="vee", color="darkorchid")

# --- Answer Generation Subgraph ---
with dot.subgraph(name="cluster_generation") as gen:
    gen.attr(label="Answer Generation", style="rounded,filled", color="seagreen", fillcolor="honeydew")
    gen.node("CON", "Concat Prompt\n+ Q Tokens", fillcolor="white")
    gen.node("T5", "T5 Model\n(Answer Generation)", fillcolor="white")
    gen.node("ANS", "Final Answer", fillcolor="white")
    gen.edge("CON", "T5", arrowhead="vee", color="seagreen")
    gen.edge("T5", "ANS", arrowhead="vee", color="seagreen")

# --- Inter-Cluster Connections ---
dot.edge("V", "CL", label="Extract Features", color="gray")
dot.edge("VB", "CM", label="Temporal Embeddings", color="gray")
dot.edge("T", "CM", label="Textual Info", style="dashed", color="gray")
dot.edge("PL", "CON", label="Visual Prompt Tokens", color="gray")
dot.edge("T", "CON", label="Q Tokens", style="dashed", color="gray")

# Render and display the diagram (output file: 'attractive_video_qa_pipeline.gv.png')
dot.render("attractive_video_qa_pipeline.gv", view=True)


'attractive_video_qa_pipeline.gv.png'

'concise_video_qa_pipeline.gv.png'